# SCARF untuk deteksi traffic Normal dan Malware

Alur notebook ini mengikuti SCARF (Bahri et al., arXiv:2106.15147v2): split train/validation/test, preprocessing, unsupervised pretraining dengan random marginal feature corruption, early stopping pada validation InfoNCE, membuang projection head, supervised fine-tuning encoder dan classification head, lalu evaluasi test dan penyimpanan artifact.

Perhatian: CSV WatchGuard ini tidak berisi ground-truth malware. Label yang dipakai adalah proxy: is_blocked=1 disebut Malware/Blocked dan is_allowed=1 disebut Normal/Allowed. Jadi hasilnya harus dibaca sebagai deteksi traffic yang diblokir sampai ada verifikasi malware dari sumber lain.

In [1]:
# 1. Setup dan konfigurasi
import ipaddress
import json
import os
import random
import warnings
from collections import Counter
from copy import deepcopy
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, average_precision_score,
    balanced_accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
TEST_SIZE = 0.20
VAL_SIZE = 0.20

# Default paper: corruption 0.6, temperature 1, batch 128, Adam 1e-3.
SCARF_BATCH_SIZE = 128
SCARF_CORRUPTION_RATE = 0.60
SCARF_TEMPERATURE = 1.0
SCARF_HIDDEN_DIM = 256
SCARF_EMBEDDING_DIM = 64
SCARF_PROJECTION_DIM = 64
SCARF_LEARNING_RATE = 1e-3
PRETRAIN_MAX_EPOCHS = 50
PRETRAIN_PATIENCE = 3
STATIC_VALIDATION_REPEATS = 10
CLASSIFIER_BATCH_SIZE = 128
CLASSIFIER_HIDDEN_DIM = 256
CLASSIFIER_LEARNING_RATE = 1e-3
CLASSIFIER_MAX_EPOCHS = 50
CLASSIFIER_PATIENCE = 3
DECISION_THRESHOLD = 0.50

def set_seed(seed=RANDOM_STATE):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

def dense_float32(matrix):
    if hasattr(matrix, 'toarray'):
        matrix = matrix.toarray()
    return np.asarray(matrix, dtype=np.float32)

def build_preprocessor(numeric_features, categorical_features):
    numeric = Pipeline([('imputer', SimpleImputer(strategy='mean')),
                        ('scaler', StandardScaler())])
    categorical = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                            ('onehot', one_hot_encoder())])
    return ColumnTransformer([('numeric', numeric, numeric_features),
                              ('categorical', categorical, categorical_features)])

def ip_features(value, prefix):
    result = {f'{prefix}_ip_is_private': 0, f'{prefix}_ip_is_global': 0,
              f'{prefix}_ip_version': 0, f'{prefix}_ip_octet1': 0,
              f'{prefix}_ip_octet2': 0}
    try:
        ip_obj = ipaddress.ip_address(str(value).strip())
        result[f'{prefix}_ip_is_private'] = int(ip_obj.is_private)
        result[f'{prefix}_ip_is_global'] = int(ip_obj.is_global)
        result[f'{prefix}_ip_version'] = int(ip_obj.version)
        if ip_obj.version == 4:
            octets = str(ip_obj).split('.')
            result[f'{prefix}_ip_octet1'] = int(octets[0])
            result[f'{prefix}_ip_octet2'] = int(octets[1])
    except ValueError:
        pass
    return result

set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATASET_FILENAME = 'syslog_dataset_2026-07-19 (1).csv'
candidates = [Path.cwd() / DATASET_FILENAME, Path('/content') / DATASET_FILENAME,
              Path('/content/drive/MyDrive/machine-learning/TA-ML') / DATASET_FILENAME,
              Path('/content/drive/MyDrive/machine-learning/TA-ML/syslog_dataset.csv')]
DATASET_PATH = next((path for path in candidates if path.exists()), None)
if DATASET_PATH is None:
    raise FileNotFoundError('CSV tidak ditemukan. Upload CSV atau ubah DATASET_FILENAME.')
drive_artifact = Path('/content/drive/MyDrive/machine-learning/TA-ML/scarf_artifacts')
ARTIFACT_DIR = drive_artifact if drive_artifact.parent.exists() else Path.cwd() / 'scarf_artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Dataset: {DATASET_PATH}')
print(f'Artifact: {ARTIFACT_DIR}')
print(f'Device: {device}')

Dataset: d:\Unduhan\model\syslog_dataset_2026-07-19 (1).csv
Artifact: d:\Unduhan\model\scarf_artifacts
Device: cpu


In [2]:
# 2. Load data dan proxy label
def normalize_flag(series):
    numeric = pd.to_numeric(series, errors='coerce')
    text = series.astype(str).str.strip().str.lower()
    mapped = text.map({'true': 1, 'yes': 1, 'y': 1, 'blocked': 1,
                       'allow': 1, 'false': 0, 'no': 0, 'n': 0,
                       'allowed': 0, 'deny': 0})
    return numeric.fillna(mapped).fillna(0).astype(int)

df_raw = pd.read_csv(DATASET_PATH)
required = {'timestamp', 'src_ip', 'dst_ip', 'protocol', 'is_blocked', 'is_allowed'}
missing = required - set(df_raw.columns)
if missing:
    raise ValueError(f'Kolom wajib tidak ditemukan: {sorted(missing)}')
data = df_raw.copy()
data['is_blocked'] = normalize_flag(data['is_blocked'])
data['is_allowed'] = normalize_flag(data['is_allowed'])
valid = (data['is_blocked'] == 1) | (data['is_allowed'] == 1)
ambiguous_count = int((~valid).sum())
data = data.loc[valid].copy()
data['target'] = (data['is_blocked'] == 1).astype(np.int64)
data['label'] = data['target'].map({0: 'Normal', 1: 'Malware'})
duplicate_count = int(data.duplicated().sum())
data = data.drop_duplicates().reset_index(drop=True)

print(f'Raw shape: {df_raw.shape}')
print(f'Baris ambigu dibuang: {ambiguous_count}')
print(f'Duplikasi exact dibuang: {duplicate_count}')
print(f'Shape setelah cleaning: {data.shape}')
print(data['label'].value_counts().to_string())
print('target=0 Normal/Allowed: is_allowed=1 dan is_blocked=0')
print('target=1 Malware/Blocked: is_blocked=1')
print('Catatan: label ini adalah proxy blocked/allowed, bukan ground-truth malware.')

Raw shape: (19998, 22)
Baris ambigu dibuang: 191
Duplikasi exact dibuang: 2279
Shape setelah cleaning: (17528, 24)
label
Normal     11964
Malware     5564
target=0 Normal/Allowed: is_allowed=1 dan is_blocked=0
target=1 Malware/Blocked: is_blocked=1
Catatan: label ini adalah proxy blocked/allowed, bukan ground-truth malware.


In [3]:
# 3. Feature engineering, split, dan preprocessing
def engineer_features(frame):
    out = frame.copy()
    for column in ['timestamp', 'hour', 'day_of_week', 'protocol', 'src_ip', 'dst_ip', 'message']:
        if column not in out.columns:
            out[column] = np.nan
    timestamp = pd.to_datetime(out['timestamp'], errors='coerce')
    hour = pd.to_numeric(out['hour'], errors='coerce').fillna(timestamp.dt.hour).fillna(0.0)
    minute = timestamp.dt.minute.fillna(0.0)
    second = timestamp.dt.second.fillna(0.0)
    day_map = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3,
               'Friday': 4, 'Saturday': 5, 'Sunday': 6}
    day = out['day_of_week'].map(day_map).fillna(timestamp.dt.dayofweek).fillna(0).astype(float)
    out['hour_sin'] = np.sin(2 * np.pi * hour / 24.0)
    out['hour_cos'] = np.cos(2 * np.pi * hour / 24.0)
    out['minute_sin'] = np.sin(2 * np.pi * minute / 60.0)
    out['minute_cos'] = np.cos(2 * np.pi * minute / 60.0)
    out['second_sin'] = np.sin(2 * np.pi * second / 60.0)
    out['second_cos'] = np.cos(2 * np.pi * second / 60.0)
    out['day_sin'] = np.sin(2 * np.pi * day / 7.0)
    out['day_cos'] = np.cos(2 * np.pi * day / 7.0)
    for column in ['src_port', 'dst_port', 'severity']:
        if column not in out.columns:
            out[column] = np.nan
    out['src_port'] = pd.to_numeric(out['src_port'], errors='coerce')
    out['dst_port'] = pd.to_numeric(out['dst_port'], errors='coerce')
    out['severity_num'] = pd.to_numeric(out['severity'], errors='coerce')
    protocol = out['protocol'].fillna('unknown').astype(str).str.lower()
    parts = protocol.str.split('/', n=1, expand=True)
    out['protocol_service'] = parts[0].replace('', 'unknown').fillna('unknown')
    out['protocol_transport'] = parts[1].replace('', 'unknown').fillna('unknown') if parts.shape[1] > 1 else 'unknown'
    src = pd.DataFrame(out['src_ip'].apply(lambda value: ip_features(value, 'src')).tolist(), index=out.index)
    dst = pd.DataFrame(out['dst_ip'].apply(lambda value: ip_features(value, 'dst')).tolist(), index=out.index)
    out = pd.concat([out, src, dst], axis=1)
    message = out['message'].fillna('unknown').astype(str)
    prefix = message.str.rsplit(')', n=1).str[-1].str.split(':', n=1).str[0].str.strip().str.lower()
    out['message_prefix'] = prefix.replace('', 'unknown').fillna('unknown')
    return out

feature_frame = engineer_features(data)
train_val_frame, test_frame = train_test_split(feature_frame, test_size=TEST_SIZE,
    random_state=RANDOM_STATE, stratify=feature_frame['target'])
val_fraction = VAL_SIZE / (1.0 - TEST_SIZE)
train_frame, val_frame = train_test_split(train_val_frame, test_size=val_fraction,
    random_state=RANDOM_STATE, stratify=train_val_frame['target'])
train_frame = train_frame.reset_index(drop=True)
val_frame = val_frame.reset_index(drop=True)
test_frame = test_frame.reset_index(drop=True)

# Label-source columns dan action dikeluarkan untuk mencegah target leakage.
numeric_candidates = ['hour_sin', 'hour_cos', 'minute_sin', 'minute_cos',
    'second_sin', 'second_cos', 'day_sin', 'day_cos', 'src_port', 'dst_port',
    'severity_num', 'src_ip_is_private', 'src_ip_is_global', 'src_ip_version',
    'src_ip_octet1', 'src_ip_octet2', 'dst_ip_is_private', 'dst_ip_is_global',
    'dst_ip_version', 'dst_ip_octet1', 'dst_ip_octet2']
categorical_candidates = ['log_type', 'event_type', 'protocol', 'protocol_service',
    'protocol_transport', 'src_country', 'dst_country', 'policy_name',
    'application', 'url_category', 'message_prefix']
numeric_features = [c for c in numeric_candidates if c in train_frame.columns and not train_frame[c].isna().all()]
categorical_features = [c for c in categorical_candidates if c in train_frame.columns and not train_frame[c].isna().all()]
feature_columns = numeric_features + categorical_features
preprocessor = build_preprocessor(numeric_features, categorical_features)
preprocessor.fit(train_frame[feature_columns])
def encode_frame(frame):
    return dense_float32(preprocessor.transform(frame[feature_columns]))
X_train_processed = encode_frame(train_frame)
X_val_processed = encode_frame(val_frame)
X_test_processed = encode_frame(test_frame)
y_train = train_frame['target'].to_numpy(dtype=np.int64)
y_val = val_frame['target'].to_numpy(dtype=np.int64)
y_test = test_frame['target'].to_numpy(dtype=np.int64)

def build_marginals(frame):
    result = {}
    for column in feature_columns:
        if column in numeric_features:
            values = pd.to_numeric(frame[column], errors='coerce').dropna().to_numpy(dtype=np.float32)
            result[column] = values if len(values) else np.array([0.0], dtype=np.float32)
        else:
            values = frame[column].dropna().astype(str).to_numpy(dtype=object)
            result[column] = values if len(values) else np.array(['unknown'], dtype=object)
    return result
marginal_values = build_marginals(train_frame)
excluded_from_features = ['is_blocked', 'is_allowed', 'action', 'label', 'target', 'message']
print(f'Train/val/test: {len(train_frame)} / {len(val_frame)} / {len(test_frame)}')
print(f'Input dimension: {X_train_processed.shape[1]}')
print(f'Numeric/categorical: {len(numeric_features)} / {len(categorical_features)}')
print(f'Label train: {Counter(y_train)} | val: {Counter(y_val)} | test: {Counter(y_test)}')
print(f'Excluded: {excluded_from_features}')

Train/val/test: 10516 / 3506 / 3506
Input dimension: 102
Numeric/categorical: 20 / 7
Label train: Counter({np.int64(0): 7178, np.int64(1): 3338}) | val: Counter({np.int64(0): 2393, np.int64(1): 1113}) | test: Counter({np.int64(0): 2393, np.int64(1): 1113})
Excluded: ['is_blocked', 'is_allowed', 'action', 'label', 'target', 'message']


In [4]:
# 4. SCARF pretraining: corruption marginal sebelum one-hot + InfoNCE
class SCARFEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, embedding_dim=64):
        super().__init__()
        self.network = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, embedding_dim))
    def forward(self, x):
        return self.network(x)

class ProjectionHead(nn.Module):
    def __init__(self, embedding_dim=64, hidden_dim=256, projection_dim=64):
        super().__init__()
        self.network = nn.Sequential(nn.Linear(embedding_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, projection_dim))
    def forward(self, x):
        return self.network(x)

class SCARFModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, embedding_dim=64, projection_dim=64):
        super().__init__()
        self.encoder = SCARFEncoder(input_dim, hidden_dim, embedding_dim)
        self.projector = ProjectionHead(embedding_dim, hidden_dim, projection_dim)
    def forward(self, x):
        embedding = self.encoder(x)
        return embedding, F.normalize(self.projector(embedding), dim=1)

def corrupt_raw_frame(frame, rng):
    source = frame[feature_columns].reset_index(drop=True).copy()
    corrupted = source.copy()
    n_rows, n_features = len(source), len(feature_columns)
    n_corrupt = max(1, int(round(SCARF_CORRUPTION_RATE * n_features)))
    mask = np.zeros((n_rows, n_features), dtype=bool)
    for row in range(n_rows):
        mask[row, rng.choice(n_features, n_corrupt, replace=False)] = True
    for index, column in enumerate(feature_columns):
        rows = np.flatnonzero(mask[:, index])
        if len(rows):
            values = marginal_values[column]
            corrupted.loc[rows, column] = values[rng.integers(0, len(values), len(rows))]
    return corrupted

def make_corrupted_matrix(frame, rng, repeats=1):
    clean = encode_frame(frame)
    views = [encode_frame(corrupt_raw_frame(frame, rng)) for _ in range(repeats)]
    return np.tile(clean, (repeats, 1)), np.vstack(views)

def infonce_loss(z_clean, z_corrupted, temperature=1.0):
    similarity = torch.matmul(z_clean, z_corrupted.T) / temperature
    targets = torch.arange(similarity.size(0), device=similarity.device)
    return F.cross_entropy(similarity, targets)

def pair_loader(clean, corrupted, shuffle):
    return DataLoader(TensorDataset(torch.tensor(clean, dtype=torch.float32),
        torch.tensor(corrupted, dtype=torch.float32)), batch_size=SCARF_BATCH_SIZE,
        shuffle=shuffle, drop_last=False)

@torch.no_grad()
def contrastive_eval(model, clean, corrupted):
    model.eval()
    total, count = 0.0, 0
    for clean_batch, corrupted_batch in pair_loader(clean, corrupted, False):
        _, z1 = model(clean_batch.to(device))
        _, z2 = model(corrupted_batch.to(device))
        loss = infonce_loss(z1, z2, SCARF_TEMPERATURE)
        total += loss.item() * len(clean_batch)
        count += len(clean_batch)
    return total / max(count, 1)

static_rng = np.random.default_rng(RANDOM_STATE + 1)
X_val_static, X_val_corrupted_static = make_corrupted_matrix(val_frame, static_rng, STATIC_VALIDATION_REPEATS)
scarf_model = SCARFModel(X_train_processed.shape[1], SCARF_HIDDEN_DIM,
    SCARF_EMBEDDING_DIM, SCARF_PROJECTION_DIM).to(device)
scarf_optimizer = torch.optim.Adam(scarf_model.parameters(), lr=SCARF_LEARNING_RATE)
pretrain_history = []
best_state, best_val_loss, wait = None, float('inf'), 0
print('Mulai SCARF pretraining...')
for epoch in range(1, PRETRAIN_MAX_EPOCHS + 1):
    rng = np.random.default_rng(RANDOM_STATE + 1000 + epoch)
    _, X_train_corrupted = make_corrupted_matrix(train_frame, rng, 1)
    scarf_model.train()
    losses = []
    for clean_batch, corrupted_batch in pair_loader(X_train_processed, X_train_corrupted, True):
        _, z1 = scarf_model(clean_batch.to(device))
        _, z2 = scarf_model(corrupted_batch.to(device))
        loss = infonce_loss(z1, z2, SCARF_TEMPERATURE)
        scarf_optimizer.zero_grad()
        loss.backward()
        scarf_optimizer.step()
        losses.append(loss.item())
    train_loss = float(np.mean(losses))
    val_loss = contrastive_eval(scarf_model, X_val_static, X_val_corrupted_static)
    pretrain_history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})
    print(f'Epoch {epoch:03d} | train InfoNCE {train_loss:.4f} | val InfoNCE {val_loss:.4f}')
    if val_loss < best_val_loss - 1e-6:
        best_val_loss = val_loss
        best_state = {name: value.detach().cpu().clone() for name, value in scarf_model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= PRETRAIN_PATIENCE:
            print(f'Early stopping pretraining pada epoch {epoch}.')
            break
if best_state is not None:
    scarf_model.load_state_dict(best_state)
scarf_model.to(device)
pretrained_path = ARTIFACT_DIR / 'scarf_pretrained.pt'
torch.save({'model_state_dict': scarf_model.state_dict(),
    'encoder_state_dict': scarf_model.encoder.state_dict(), 'input_dim': int(X_train_processed.shape[1]),
    'embedding_dim': SCARF_EMBEDDING_DIM, 'history': pretrain_history}, pretrained_path)
print(f'Best validation InfoNCE: {best_val_loss:.4f}')
print(f'Pretrained encoder tersimpan di: {pretrained_path}')

Mulai SCARF pretraining...
Epoch 001 | train InfoNCE 4.4361 | val InfoNCE 4.3918
Epoch 002 | train InfoNCE 4.3538 | val InfoNCE 4.3712
Epoch 003 | train InfoNCE 4.3469 | val InfoNCE 4.3613
Epoch 004 | train InfoNCE 4.3218 | val InfoNCE 4.3343
Epoch 005 | train InfoNCE 4.3128 | val InfoNCE 4.3330
Epoch 006 | train InfoNCE 4.3081 | val InfoNCE 4.3251
Epoch 007 | train InfoNCE 4.3091 | val InfoNCE 4.3175
Epoch 008 | train InfoNCE 4.2894 | val InfoNCE 4.3105
Epoch 009 | train InfoNCE 4.2937 | val InfoNCE 4.3131
Epoch 010 | train InfoNCE 4.2864 | val InfoNCE 4.3033
Epoch 011 | train InfoNCE 4.2853 | val InfoNCE 4.2963
Epoch 012 | train InfoNCE 4.2745 | val InfoNCE 4.2886
Epoch 013 | train InfoNCE 4.2707 | val InfoNCE 4.2867
Epoch 014 | train InfoNCE 4.2589 | val InfoNCE 4.2837
Epoch 015 | train InfoNCE 4.2569 | val InfoNCE 4.2747
Epoch 016 | train InfoNCE 4.2540 | val InfoNCE 4.2730
Epoch 017 | train InfoNCE 4.2527 | val InfoNCE 4.2707
Epoch 018 | train InfoNCE 4.2541 | val InfoNCE 4.2721
E

In [5]:
# 5. Fine-tuning supervised: projection head dibuang, encoder ikut di-update
class SCARFClassifier(nn.Module):
    def __init__(self, encoder, embedding_dim=64, hidden_dim=256):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(nn.Linear(embedding_dim, hidden_dim), nn.ReLU(),
                                  nn.Linear(hidden_dim, 2))
    def forward(self, x):
        return self.head(self.encoder(x))

def supervised_loader(X, y, shuffle):
    return DataLoader(TensorDataset(torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long)), batch_size=CLASSIFIER_BATCH_SIZE, shuffle=shuffle)

@torch.no_grad()
def predict(model, loader):
    model.eval()
    ys, preds, probs = [], [], []
    for X_batch, y_batch in loader:
        probability = F.softmax(model(X_batch.to(device)), dim=1).cpu().numpy()
        ys.append(y_batch.numpy())
        preds.append(np.argmax(probability, axis=1))
        probs.append(probability)
    return np.concatenate(ys), np.concatenate(preds), np.vstack(probs)

classifier_model = SCARFClassifier(scarf_model.encoder, SCARF_EMBEDDING_DIM,
    CLASSIFIER_HIDDEN_DIM).to(device)
optimizer = torch.optim.Adam(classifier_model.parameters(), lr=CLASSIFIER_LEARNING_RATE)
criterion = nn.CrossEntropyLoss()
train_loader = supervised_loader(X_train_processed, y_train, True)
val_loader = supervised_loader(X_val_processed, y_val, False)
classifier_history = []
best_state, best_error, wait = None, float('inf'), 0
print('Mulai supervised fine-tuning...')
for epoch in range(1, CLASSIFIER_MAX_EPOCHS + 1):
    classifier_model.train()
    losses = []
    for X_batch, y_batch in train_loader:
        loss = criterion(classifier_model(X_batch.to(device)), y_batch.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    y_val_true, y_val_pred, _ = predict(classifier_model, val_loader)
    val_accuracy = accuracy_score(y_val_true, y_val_pred)
    val_error = 1.0 - val_accuracy
    classifier_history.append({'epoch': epoch, 'train_loss': float(np.mean(losses)),
                               'val_accuracy': float(val_accuracy), 'val_error': float(val_error)})
    print(f'Epoch {epoch:03d} | train CE {np.mean(losses):.4f} | val accuracy {val_accuracy:.4f}')
    if val_error < best_error - 1e-6:
        best_error = val_error
        best_state = {name: value.detach().cpu().clone() for name, value in classifier_model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= CLASSIFIER_PATIENCE:
            print(f'Early stopping fine-tuning pada epoch {epoch}.')
            break
if best_state is not None:
    classifier_model.load_state_dict(best_state)
classifier_model.to(device)
print(f'Best validation accuracy: {1.0 - best_error:.4f}')

Mulai supervised fine-tuning...
Epoch 001 | train CE 0.1078 | val accuracy 0.9977
Epoch 002 | train CE 0.0091 | val accuracy 0.9989
Epoch 003 | train CE 0.0050 | val accuracy 0.9974
Epoch 004 | train CE 0.0032 | val accuracy 1.0000
Epoch 005 | train CE 0.0037 | val accuracy 1.0000
Epoch 006 | train CE 0.0029 | val accuracy 1.0000
Epoch 007 | train CE 0.0026 | val accuracy 1.0000
Early stopping fine-tuning pada epoch 7.
Best validation accuracy: 1.0000


In [6]:
# 6. Evaluasi final, artifact, dan fungsi inference
test_loader = supervised_loader(X_test_processed, y_test, False)
y_true, y_pred, y_prob = predict(classifier_model, test_loader)
malware_prob = y_prob[:, 1]
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred,
    average='binary', pos_label=1, zero_division=0)
metrics = {'accuracy': float(accuracy_score(y_true, y_pred)),
    'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
    'precision_malware': float(precision), 'recall_malware': float(recall),
    'f1_malware': float(f1), 'roc_auc': float(roc_auc_score(y_true, malware_prob)),
    'pr_auc': float(average_precision_score(y_true, malware_prob))}
print('=' * 60)
print('HASIL FINAL PADA TEST SET')
print('=' * 60)
print('Confusion matrix [Normal, Malware]:')
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))
print(classification_report(y_true, y_pred, labels=[0, 1],
    target_names=['Normal', 'Malware'], digits=4, zero_division=0))
print('Metrics:', metrics)

def predict_new_dataframe(raw_frame, threshold=DECISION_THRESHOLD):
    already_engineered = set(feature_columns).issubset(raw_frame.columns) and not raw_frame.columns.duplicated().any()
    engineered = raw_frame if already_engineered else engineer_features(raw_frame)
    X_new = encode_frame(engineered)
    loader = DataLoader(TensorDataset(torch.tensor(X_new, dtype=torch.float32)),
                        batch_size=CLASSIFIER_BATCH_SIZE, shuffle=False)
    classifier_model.eval()
    probabilities = []
    with torch.no_grad():
        for (X_batch,) in loader:
            probabilities.append(F.softmax(classifier_model(X_batch.to(device)), dim=1).cpu().numpy())
    probabilities = np.vstack(probabilities)
    result = raw_frame.copy().reset_index(drop=True)
    result['malware_probability'] = probabilities[:, 1]
    result['prediction'] = np.where(result['malware_probability'] >= threshold, 'Malware', 'Normal')
    return result

test_predictions = predict_new_dataframe(test_frame)
test_predictions['target'] = y_true
test_predictions['predicted_target'] = y_pred
test_predictions.to_csv(ARTIFACT_DIR / 'test_predictions.csv', index=False)
pd.DataFrame(pretrain_history).to_csv(ARTIFACT_DIR / 'pretrain_history.csv', index=False)
pd.DataFrame(classifier_history).to_csv(ARTIFACT_DIR / 'classifier_history.csv', index=False)
joblib.dump(preprocessor, ARTIFACT_DIR / 'preprocessor.joblib')
metadata = {'dataset_path': str(DATASET_PATH),
    'label_definition': {'0': 'Normal/Allowed proxy', '1': 'Malware/Blocked proxy'},
    'excluded_from_features': excluded_from_features,
    'numeric_features': numeric_features, 'categorical_features': categorical_features,
    'feature_columns': feature_columns, 'input_dim': int(X_train_processed.shape[1]),
    'embedding_dim': SCARF_EMBEDDING_DIM, 'decision_threshold': DECISION_THRESHOLD,
    'metrics': metrics, 'split_sizes': {'train': len(train_frame), 'validation': len(val_frame), 'test': len(test_frame)},
    'scarf': {'corruption_rate': SCARF_CORRUPTION_RATE, 'temperature': SCARF_TEMPERATURE,
              'batch_size': SCARF_BATCH_SIZE, 'pretrain_patience': PRETRAIN_PATIENCE}}
(ARTIFACT_DIR / 'metadata.json').write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')
torch.save({'model_state_dict': classifier_model.state_dict(),
    'input_dim': int(X_train_processed.shape[1]), 'embedding_dim': SCARF_EMBEDDING_DIM,
    'hidden_dim': CLASSIFIER_HIDDEN_DIM, 'metrics': metrics}, ARTIFACT_DIR / 'scarf_classifier.pt')
print(f'Artifact tersimpan di: {ARTIFACT_DIR}')
display(test_predictions[['malware_probability', 'prediction', 'target', 'predicted_target']].head(10))

HASIL FINAL PADA TEST SET
Confusion matrix [Normal, Malware]:
[[2393    0]
 [   3 1110]]
              precision    recall  f1-score   support

      Normal     0.9987    1.0000    0.9994      2393
     Malware     1.0000    0.9973    0.9987      1113

    accuracy                         0.9991      3506
   macro avg     0.9994    0.9987    0.9990      3506
weighted avg     0.9991    0.9991    0.9991      3506

Metrics: {'accuracy': 0.9991443240159726, 'balanced_accuracy': 0.9986522911051212, 'precision_malware': 1.0, 'recall_malware': 0.9973045822102425, 'f1_malware': 0.9986504723346828, 'roc_auc': 0.9997786671142134, 'pr_auc': 0.9996184249736936}
Artifact tersimpan di: d:\Unduhan\model\scarf_artifacts


,malware_probability,prediction,target,predicted_target
0,9.999675e-01,Malware,1,1
1,1.314087e-07,Normal,0,0
2,2.115369e-08,Normal,0,0
3,9.999999e-01,Malware,1,1
4,9.999918e-01,Malware,1,1
5,4.033355e-06,Normal,0,0
6,1.133808e-06,Normal,0,0
7,9.999998e-01,Malware,1,1
8,1.699877e-05,Normal,0,0
9,2.027610e-07,Normal,0,0


## Inference pada log baru

Setelah cell training selesai, jalankan contoh berikut untuk CSV WatchGuard baru. Preprocessor hanya ditransform, tidak di-fit ulang.

In [7]:
# new_logs = pd.read_csv('watchguard_logs_baru.csv')
# detections = predict_new_dataframe(new_logs, threshold=DECISION_THRESHOLD)
# display(detections[['timestamp', 'malware_probability', 'prediction']].head())